In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Data Reading from Silver Layer

In [0]:
df = spark.read.table('medallion_arc_e2e.silver.products')

display(df)

# To See Duplicated Rows

In [0]:
df.columns

In [0]:
df.groupBy(df.columns).count().filter(col('count') > 1).show()

# Removing Duplicates

In [0]:
df = df.dropDuplicates(subset=['product_id'])

display(df.limit(10))

# Dividing New vs Old Records

# Define df_new (from silver layer)

In [0]:
df_new = spark.read.table('medallion_arc_e2e.silver.products')

In [0]:
if spark.catalog.tableExists('medallion_arc_e2e.gold.products'):
    df_old = spark.read.table('medallion_arc_e2e.gold.dim_products')

else:
    df_old = spark.createDataFrame([], df_new.schema)

In [0]:
display(df_old)

# Add metadata columns (important for SCD2)

In [0]:
# from pyspark.sql.functions import current_timestamp, lit

df_new = df_new.withColumn("updated_at", current_timestamp())

# SCD Type 1 Logic (Overwrite changes)

In [0]:
from pyspark.sql.functions import col

df_scd1 = df_new.alias("new").join(
    df_old.alias("old"),
    on="product_id",
    how="left"
).select("new.*")

# 👉 This essentially replaces old records

In [0]:
display(df_scd1)

# SCD Type 2 Logic (Track history)
## Step 1: Add SCD2 columns

In [0]:
df_old = df_old.withColumn("is_current", col("is_current")) \
               .withColumn("start_date", col("start_date")) \
               .withColumn("end_date", col("end_date"))

## Step 2: Identify changed records

In [0]:
df_join = df_new.alias("new").join(
    df_old.alias("old"),
    on="product_id",
    how="left"
)

df_changed = df_join.filter(
    (col("old.product_id").isNotNull()) &
    (col("new.product_name") != col("old.product_name"))
)

## Step 3: Expire old records

In [0]:
df_expired = df_changed.select("old.*") \
    .withColumn("is_current", lit(False)) \
    .withColumn("end_date", current_timestamp())

## Step 4: Insert new records

In [0]:
df_new_records = df_changed.select("new.*") \
    .withColumn("is_current", lit(True)) \
    .withColumn("start_date", current_timestamp()) \
    .withColumn("end_date", lit(None))

## Step 5: Handle brand new records

In [0]:
df_inserts = df_join.filter(col("old.product_id").isNull()) \
    .select("new.*") \
    .withColumn("is_current", lit(True)) \
    .withColumn("start_date", current_timestamp()) \
    .withColumn("end_date", lit(None))

# Final Upsert using MERGE (Best Practice)

In [0]:
%sql

MERGE INTO medallion_arc_e2e.gold.dim_products AS target
USING temp_view_products AS source
ON target.product_id = source.product_id

WHEN MATCHED AND target.is_current = true AND (
    target.product_name <> source.product_name
) THEN
  UPDATE SET
    target.is_current = false,
    target.end_date = current_timestamp()

WHEN NOT MATCHED THEN
  INSERT *

In [0]:
# table_name = "medallion_arc_e2e.gold.dim_products"

# if not spark.catalog.tableExists(table_name):

#     df_new.write.format("delta") \
#     .mode("overwrite") \
#     .saveAsTable(table_name)

# else:
#     from delta.tables import DeltaTable

#     delta_table = DeltaTable.forName(spark, table_name)

#     delta_table.alias("target").merge(
#         df_new.alias("source"),
#         "target.product_id = source.product_id"
#     ).whenMatchedUpdate(
#         condition="""
#             target.product_name <> source.product_name 
#             AND target.is_current = true
#         """,
#         set={
#             "is_current": "false",
#             "end_date": "current_timestamp()"
#         }
#     ).whenNotMatchedInsertAll().execute()

In [0]:
table_name = "medallion_arc_e2e.gold.dim_products"

from pyspark.sql.functions import current_timestamp, lit

df_new = df_new \
    .withColumn("start_date", current_timestamp()) \
    .withColumn("end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

if not spark.catalog.tableExists(table_name):
    print("🔹 Initial Load")
    
    df_new.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

else:
    print("🔹 Incremental Load (MERGE)")
    
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, table_name)

    delta_table.alias("target").merge(
        df_new.alias("source"),
        "target.product_id = source.product_id"
    ).whenMatchedUpdate(
        condition="""
            target.product_name <> source.product_name 
            AND target.is_current = true
        """,
        set={
            "is_current": "false",
            "end_date": "current_timestamp()"
        }
    ).whenNotMatchedInsertAll().execute()

In [0]:
# from delta.tables import DeltaTable

# delta_table = DeltaTable.forName(spark, "medallion_arc_e2e.gold.dim_products")

# delta_table.alias("target").merge(
#     df_new.alias("source"),
#     "target.product_id = source.product_id"
# ).whenMatchedUpdate(
#     condition="target.product_name <> source.product_name AND target.is_current = true",
#     set={
#         "is_current": "false",
#         "end_date": "current_timestamp()"
#     }
# ).whenNotMatchedInsertAll().execute()

In [0]:
%sql
Select * from medallion_arc_e2e.gold.dim_products;